In [6]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

%store -r MIMIC_DIR
%store -r cohort

In [7]:
# CANCER STAGES / METASTATIC STATUS
%store -r cancer_matches
%store -r Diagnoses
metastatic_mask = ( # looking for metastatic cancer
    ((Diagnoses.icd_version == 9) & Diagnoses.icd_code.str.match(r"^19[678]")) |
    ((Diagnoses.icd_version == 10) & Diagnoses.icd_code.str.match(r"^C7[789]"))
)

metastatic_flag = ( # getting rid of duplicates and reassigning
    Diagnoses[metastatic_mask][["hadm_id"]]
    .drop_duplicates()
    .assign(has_metastatic_disease=True)
)
%store metastatic_flag

Stored 'metastatic_flag' (DataFrame)


In [8]:
cancer_categories = {
    "head_neck":    {"icd9": r"^14[0-9]",            "icd10": r"^C(0[0-9]|1[0-4])"},
    "gi":           {"icd9": r"^1(5[0-9]|6[0-9])",   "icd10": r"^C(1[5-9]|2[0-6])"},
    "lung_thoracic":{"icd9": r"^16[2-5]",            "icd10": r"^C(3[3-9])"},
    "bone_softtissue":{"icd9": r"^17[0-1]",          "icd10": r"^C(40|41|4[5-9])"},
    "skin_melanoma":{"icd9": r"^17[2-3]",            "icd10": r"^C4[34]"},
    "breast":       {"icd9": r"^174",                "icd10": r"^C50"},
    "gu":           {"icd9": r"^1(79|8[0-9])",       "icd10": r"^C(5[1-9]|6[0-3])"},
    "renal_bladder":{"icd9": r"^18[8-9]",            "icd10": r"^C(64|65|66|67|68)"},
    "cns":          {"icd9": r"^19[1-2]",            "icd10": r"^C(7[0-2])"},
    "endocrine":    {"icd9": r"^193|194",            "icd10": r"^C7[34]"},
    "heme_lymphoma":{"icd9": r"^20[0-8]",            "icd10": r"^C(8[1-9]|9[0-6])"},
    "metastatic_unspecified":{"icd9": r"^19[5-9]",   "icd10": r"^C7[6-9]|C80"},
}
%store cancer_categories

Stored 'cancer_categories' (dict)


In [9]:
def flag_cancer_type(diagnoses_df, categories):
    out = pd.DataFrame({"hadm_id": diagnoses_df["hadm_id"].unique()})
    for label, patterns in categories.items():
        mask = (
            ((diagnoses_df.icd_version == 9) & diagnoses_df.icd_code.str.match(patterns["icd9"])) |
            ((diagnoses_df.icd_version == 10) & diagnoses_df.icd_code.str.match(patterns["icd10"]))
        )
        flagged_ids = set(diagnoses_df.loc[mask, "hadm_id"])
        out[f"cancer_{label}"] = out["hadm_id"].isin(flagged_ids)
    return out
cancer_type_flags = flag_cancer_type(Diagnoses, cancer_categories)
%store cancer_type_flags

Stored 'cancer_type_flags' (DataFrame)


In [10]:
cancer_type_flags["n_cancer_types_flagged"] = cancer_type_flags.filter(like = "cancer_").sum(axis=1)
%store cancer_type_flags

Stored 'cancer_type_flags' (DataFrame)
